In [1]:
from config import llm
from typing import List, Optional, Annotated, Dict, Any, Literal
import operator
from pydantic import BaseModel, Field
from typing_extensions import TypedDict
from enum import Enum
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [2]:
class SearchStep(BaseModel):
    step: int
    strategy: Literal["metadata_filter", "semantic_query", "sorting", "select_top_x"]
    parameters: Dict[str, Any] = Field(description="Parámetros como {'year': 2024} o {'query': '...'} ")
    justification: str

class PlanningOutput(BaseModel):
    intent: Literal["LIST_COUNT", "ELABORATE"]
    plan: List[SearchStep]
    generated_queries: List[str] = Field(description="2 variaciones semánticas sin nombres de empresas")
    redefined_task: str

In [3]:
from typing import List, Optional, Annotated, Dict, Any, Literal
import operator
from pydantic import BaseModel, Field
from typing_extensions import TypedDict
from enum import Enum


# --- ENUMS Y MODELOS DE APOYO ---

class IntentEnum(str, Enum):
    QUALITATIVE = "qualitative"
    LISTING = "listing"
    QUANTITATIVE = "quantitative"

class MetadataFilter(BaseModel):
    year: Optional[int] = Field(None, description="Año específico del post si se menciona (ej: 2024, 2026)")   
    
class ActionStep(BaseModel):
    step_number: int = Field(..., description="Orden secuencial del paso")
    action: Literal[
        "METADATA_FILTER", 
        "SEMANTIC_SEARCH", 
        "SORT_DESC", 
        "SELECT_TOP",
        "COUNT_RESULTS", 
        "EXTRACT_INFO"
    ] = Field(..., description="Comando técnico permitido")
    limit: Optional[int] = Field(
        None, 
        description="Número de elementos a retornar (usar solo en SELECT_TOP o si la búsqueda requiere un top N)"
    )
    description: str = Field(..., description="Explicación breve de qué hace este paso")

# --- MODELO DE SALIDA DEL ANALIZADOR (LLM) ---

class QueryAnalysis(BaseModel):
    intent: IntentEnum = Field(..., description="Categoría del intent")    
    semantic_queries: List[str] = Field(
        ..., 
        min_length=2, 
        max_length=3, 
        description="Variaciones de la búsqueda"
    )
    action_plan: List[ActionStep] = Field(..., description="Pasos secuenciales")
    refined_task: str = Field(..., description="Instrucción para el Generator")
    metadata_filter: MetadataFilter = Field(..., description="Filtros extraídos")
    reasoning: str = Field(..., description="Justificación lógica")
    entities: List[str] = Field(default_factory=list, description="Keywords extraídas")

# --- ESTADO DEL AGENTE (LANGGRAPH) ---

class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    intent: str
    metadata_filter: Dict[str, Any]
    retrieved_docs: List[Any]  # O List[Document]
    generation: str
    loop_step: int
    attempts: int
    is_out_of_domain: bool
    analysis: QueryAnalysis
    current_step_idx: int

In [4]:
SYSTEM_INSTRUCTION= """
### ROLE
Eres un experto en Arquitectura de Recuperación y Generación (RAG). 
Tu función es actuar como el "Nodo de Planificación" que analiza la 
consulta del usuario antes de ejecutar cualquier búsqueda.
Debes definir qué se busca y cómo se responderá.

### STEP 1: CLASIFICACIÓN DE INTENCIÓN (intent)
Debes identificar qué tipo de respuesta se espera, independientemente de cómo se recupere la información:

1. **LIST_COUNT**: El usuario busca un conteo o un listado de documentos o metadatos (links, títulos). 
Normalmente NO se requiere leer el contenido interno para generar la respuesta.
   *Ejemplos:* "Dime los links de los últimos 3 posts", "¿Cuántos artículos hay de IA?", "Lista los blogs de DevOps de 2024".

2. **ELABORATE**: El usuario requiere una respuesta elaborada basada en el contenido.
SI o SI se debe leer el interior de los documentos para generar una respuesta.
   *Ejemplos:* "¿Cuánta azúcar lleva una pastafrola?", "Explica el enfoque de RAG de la empresa", "¿Qué requisitos de memoria tiene Llama3?".

### STEP 2: ESTRATEGIA DE RECUPERACIÓN (plan)
Define una secuencia de pasos (`SearchStep`) para encontrar la información necesaria. Las estrategias permitidas son:
- **metadata_filter**: Uso de datos estructurados (Actualmente solo disponible: `year`).
- **semantic_query**: Búsqueda por significado/conceptos. Úsala cuando el año no sea suficiente para acotar el tema (Ej: "que es un LLM?").
- **sorting**: Ordenar resultados (TOP, BOTTOM) basado en tiempo o relevancia.
- **select_top_x**: Definir cuántos documentos (`X`) son necesarios recuperar.

### REGLAS DE PRIVACIDAD Y REESCRITURA
- **generated_queries**: Crea 2 nuevas búsquedas semánticas para mejorar el recall.
-**PROHIBIDO** incluir el nombre "Bitovi"; reemplázalo por términos genéricos como "empresa", "blog técnico" o elimínalo.
- **redefined_task**: Instrucción final para el generador. No debe contener el nombre "Bitovi".

### EJEMPLOS DE REFERENCIA (FEW-SHOT)

User: "¿De qué trata el último blog de Bitovi?"
AI: {{
  "intent": "qualitative",
  "action_plan": [
    {{"step_number": 1, "action": "SORT_DESC", "description": "Ordenar por fecha descendente para identificar el documento más reciente."}},
    {{"step_number": 2, "action": "SELECT_TOP", "limit": 1, "description": "Seleccionar la entrada más nueva para su análisis."}},
    
  ],
  "metadata_filter": {{"year": null}},
  "semantic_queries": [
    "Latest technical implementations and engineering updates",
    "Current software architecture patterns and autonomous systems",
    "Cutting-edge developments in RAG and agentic workflows"
  ],
  "refined_task": "Summarize the technical subject of the provided context from the knowledge base, describing tools and architecture.",
  "reasoning": "Se clasifica como qualitative porque pide 'de qué trata' (contenido). Se anonimiza la fuente y se prioriza el orden cronológico.",
  "entities": ["último blog"]
}}

User: "Cuanta memoria necesita llama3:8b?"
AI: {{
  "intent": "qualitative",
  "action_plan": [
    {{"step_number": 1, "action": "SEMANTIC_SEARCH", "description": "Buscar requisitos técnicos de hardware para llama3:8b."}},
    {{"step_number": 2, "action": "SELECT_TOP", "limit": 3, "description": "Extraer los 3 fragmentos más relevantes sobre consumo de memoria."}}
  ],
  "metadata_filter": {{"year": null}},
  "semantic_queries": [
    "Llama3 8b RAM VRAM system requirements",
    "Inference hardware specifications for small language models",
    "Memory consumption and parameter quantization for 8b models"
  ],
  "refined_task": "Based on the provided context, determine the memory requirements for llama3:8b.",
  "reasoning": "Es qualitative porque busca un dato técnico específico (RAM/VRAM) dentro del contenido. No hay filtro de año.",
  "entities": ["llama3:8b", "memoria"]
}}

User: "¿Cuántos posts de IA hay en 2024?"
AI: {{
  "intent": "quantitative",
  "action_plan": [
    {{"step_number": 1, "action": "METADATA_FILTER", "description": "Aplicar filtro por año 2024."}},
    {{"step_number": 2, "action": "SEMANTIC_SEARCH", "description": "Identificar documentos relacionados con inteligencia artificial."}},
    {{"step_number": 3, "action": "COUNT_RESULTS", "description": "Contar el total de registros que cumplen ambos criterios."}}
  ],
  "metadata_filter": {{"year": 2024}},
  "semantic_queries": [
    "Artificial Intelligence and Machine Learning implementations",
    "Neural networks and large language model integration"
  ],
  "refined_task": "Count and report the total number of articles related to AI and Machine Learning from 2024 present in the engineering blog.",
  "reasoning": "Intent quantitative por conteo de archivos. Se incluye filtro de metadata explícito por el año 2024.",
  "entities": ["IA", "2024"]
}}

User: "Dame los últimos 5 posts de DevOps"
AI: {{
  "intent": "listing",
  "action_plan": [
    {{"step_number": 1, "action": "SEMANTIC_SEARCH", "description": "Filtrar por la temática técnica de infraestructura y automatización."}},
    {{"step_number": 2, "action": "SORT_DESC", "description": "Ordenar los resultados por fecha de publicación."}},
    {{"step_number": 3, "action": "SELECT_TOP", "limit": 5, "description": "Extraer los 5 elementos superiores."}}
  ],
  "metadata_filter": {{"year": null}},
  "semantic_queries": [
    "Continuous integration and deployment pipeline automation",
    "Infrastructure as code and cloud orchestration strategies",
    "Site reliability engineering and monitoring practices"
  ],
  "refined_task": "List the titles and links of the 5 most recent DevOps articles in the knowledge base.",
  "reasoning": "Intent listing para recuperación de una colección. Se aplica búsqueda semántica primero para asegurar relevancia temática antes del ordenamiento.",
  "entities": ["últimos 5", "DevOps"]
}}


"""

In [8]:
planner_node = llm.with_structured_output(PlanningOutput)

user_query = "¿Cuantos articulos sobre circulacion forzada hay en el blog de Bitovi escritos en 2023?"
# user_query = "What is the latest blog about?"

prompt = [
    SystemMessage(content=SYSTEM_INSTRUCTION),
    HumanMessage(content=user_query)
]

# 4. Ejecución
# Al usar with_structured_output, 'resultado' es una instancia de PlanningOutput
resultado = planner_node.invoke(prompt)

# 5. Acceso a los datos (Sin errores de AttributeError)
print(f"Intención: {resultado.intent}")
print(f"Pasos del plan: {len(resultado.plan)}")
print(f"Nueva instrucción: {resultado.redefined_task}")

# Si necesitás convertirlo a diccionario para un estado de LangGraph:
state_update = resultado.model_dump()

Intención: ELABORATE
Pasos del plan: 4
Nueva instrucción: Summarize the number of articles about forced circulation in Bitovi's blog written in 2023, including a brief description of each article and its relevance to the topic.


In [7]:
state_update

{'intent': 'ELABORATE',
 'plan': [{'step': 1,
   'strategy': 'metadata_filter',
   'parameters': {'year': None},
   'justification': 'Identificar el último post'},
  {'step': 2,
   'strategy': 'semantic_query',
   'parameters': {'query': 'latest blog'},
   'justification': 'Buscar contenido relacionado con el último post'}],
 'generated_queries': ['Latest technical implementations and engineering updates',
  'Current software architecture patterns and autonomous systems'],
 'redefined_task': 'Summarize the content of the latest blog from the knowledge base, describing tools and architecture.'}